# 04 · Features and model configuration

A `Feature` describes one input column and what (if anything) to predict from it. The model
builds its embeddings, heads and losses from this list.

## The four kinds of feature

| You have | `Feature(...)` | Output | Loss |
|---|---|---|---|
| categorical input, predict the next token | `Feature("item", "classification", vocab_size=V)` | `(N, V)` logits | cross entropy |
| numeric input, predict the next value | `Feature("price", "regression", is_linear=True)` | `(N, 1)` | MSE |
| numeric input, predict a bucket | `Feature("qty", "classification", is_linear=True, n_classes=3)` | `(N, 3)` logits | cross entropy |
| input only | `Feature("weekday", "passthrough", vocab_size=8)` | none | none |

- `vocab_size`: number of token ids of a **categorical input**, including PAD (0) and UNK (1).
- `is_linear=True`: a **numeric input**, embedded with `nn.Linear(1, d)`.
- `n_classes`: number of classes of a **classification target**; defaults to `vocab_size`.

In [1]:
from next_sequence import Feature, TaskType

features = [
    Feature("item", TaskType.CLASSIFICATION, vocab_size=7),
    Feature("price", TaskType.REGRESSION, is_linear=True),
    Feature("qty", "classification", is_linear=True, n_classes=3),   # strings work too
    Feature("weekday", "passthrough", vocab_size=8),
]
for f in features:
    print(f)

Feature(name='item', task=<TaskType.CLASSIFICATION: 'classification'>, vocab_size=7, is_linear=False, n_classes=7)
Feature(name='price', task=<TaskType.REGRESSION: 'regression'>, vocab_size=0, is_linear=True, n_classes=None)
Feature(name='qty', task=<TaskType.CLASSIFICATION: 'classification'>, vocab_size=0, is_linear=True, n_classes=3)
Feature(name='weekday', task=<TaskType.PASSTHROUGH: 'passthrough'>, vocab_size=8, is_linear=False, n_classes=None)


## Mistakes are caught immediately

Invalid combinations raise a `ValueError` with a message that says what to change.

In [2]:
attempts = {
    "categorical input without vocab_size": dict(name="item", task="classification"),
    "numeric classification without n_classes": dict(name="qty", task="classification", is_linear=True),
    "n_classes on a regression": dict(name="price", task="regression", is_linear=True, n_classes=3),
    "a dot in the name": dict(name="item.id", task="regression", is_linear=True),
}
for label, kwargs in attempts.items():
    try:
        Feature(**kwargs)
    except ValueError as e:
        print(f"{label:42s} -> {e}")

categorical input without vocab_size       -> item: a categorical input needs vocab_size >= 2 (including PAD). For a numeric input, set is_linear=True.
numeric classification without n_classes   -> qty: classification needs n_classes >= 2, got 0
n_classes on a regression                  -> price: n_classes is only used with task=classification
a dot in the name                          -> Feature name must not contain '.', got 'item.id'


## Model configuration

`NextSequenceConfig` holds the architecture. The width of the transformer is `fusion_dim`
(default: `emb_dim`), and `fusion_dim / attn_heads` must be an even integer (for RoPE).

In [3]:
from next_sequence import NextSequenceConfig

config = NextSequenceConfig(features=features, max_len=16, emb_dim=32, attn_heads=2, num_layers=2)
config

NextSequenceConfig(features=[Feature(name='item', task=<TaskType.CLASSIFICATION: 'classification'>, vocab_size=7, is_linear=False, n_classes=7), Feature(name='price', task=<TaskType.REGRESSION: 'regression'>, vocab_size=0, is_linear=True, n_classes=None), Feature(name='qty', task=<TaskType.CLASSIFICATION: 'classification'>, vocab_size=0, is_linear=True, n_classes=3), Feature(name='weekday', task=<TaskType.PASSTHROUGH: 'passthrough'>, vocab_size=8, is_linear=False, n_classes=None)], max_len=16, emb_dim=32, fusion_dim=32, fusion_strategy=<FusionStrategy.SUM: 'sum'>, rope_strategy=<RoPEStrategy.INTERLEAVED: 'interleaved'>, attn_heads=2, num_layers=2, expansion=2, dropout=0.1)

In [4]:
bad_configs = {
    "fusion_dim not divisible by heads": dict(emb_dim=30, attn_heads=4),
    "odd head width": dict(emb_dim=18, attn_heads=2),
    "duplicate feature": dict(features=features + [features[0]]),
    "nothing to predict": dict(features=[Feature("weekday", "passthrough", vocab_size=8)]),
}
for label, kwargs in bad_configs.items():
    try:
        NextSequenceConfig(**{"features": features, "max_len": 16, **kwargs})
    except ValueError as e:
        print(f"{label:36s} -> {e}")

fusion_dim not divisible by heads    -> fusion_dim (30) must be divisible by attn_heads (4)
odd head width                       -> fusion_dim / attn_heads = 9 must be even (RoPE)
duplicate feature                    -> duplicate feature names: ['item']
nothing to predict                   -> at least one feature must be predicted (not PASSTHROUGH)


A config is plain data, so it can be saved as JSON and rebuilt:

In [5]:
import json

as_json = json.dumps(config.to_dict(), indent=1)
print(as_json[:300], "...")
assert NextSequenceConfig.from_dict(json.loads(as_json)) == config

{
 "schema_version": 1,
 "features": [
  {
   "name": "item",
   "task": "classification",
   "vocab_size": 7,
   "is_linear": false,
   "n_classes": 7
  },
  {
   "name": "price",
   "task": "regression",
   "vocab_size": 0,
   "is_linear": true,
   "n_classes": null
  },
  {
   "name": "qty",
   " ...


## What the model looks like

`NextSequenceModel` is a plain `nn.Module`:
embeddings → fusion → CLS token + transformer encoder → one head per predicted feature.

In [6]:
from next_sequence import NextSequenceModel

model = NextSequenceModel(config)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
model

18,155 parameters


NextSequenceModel(
  (backbone): EncoderCLSBackbone(
    (embedding): EmbeddingModel(
      (embedder): ModuleDict(
        (item): Embedding(7, 32, padding_idx=0)
        (price): Linear(in_features=1, out_features=32, bias=True)
        (qty): Linear(in_features=1, out_features=32, bias=True)
        (weekday): Embedding(8, 32, padding_idx=0)
      )
    )
    (fusion): SumFusion(
      (proj): Identity()
    )
    (prepend_cls): PrependCLS()
    (rope): RoPE()
    (layers): ModuleList(
      (0-1): 2 x EncoderBlock(
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): MultiHeadAttention(
          (wQ): Linear(in_features=32, out_features=32, bias=True)
          (wK): Linear(in_features=32, out_features=32, bias=True)
          (wV): Linear(in_features=32, out_features=32, bias=True)
          (wO): Linear(in_features=32, out_features=32, bias=True)
          (rope): RoPE()
        )
        (norm2): LayerNorm((32,), eps=1e-05, elementwis

## A forward pass on a fake batch

Categorical inputs are `(B, T)` token ids, numeric inputs `(B, T, 1)`. Outputs are raw:
logits for classification, values for regression. The passthrough feature has no output.

In [7]:
import torch

B, T = 4, 6
X = {"item": torch.randint(2, 7, (B, T)), "price": torch.randn(B, T, 1),
     "qty": torch.randn(B, T, 1), "weekday": torch.randint(1, 8, (B, T))}
mask = torch.zeros(B, T, dtype=torch.bool)
mask[0, 4:] = True                              # the first sequence has 2 padded steps

model.eval()
with torch.no_grad():
    out = model(X, mask)
{name: tuple(t.shape) for name, t in out.items()}

{'item': (4, 7), 'price': (4, 1), 'qty': (4, 3)}

## Combining features: `sum` vs `concat`

- `sum` adds the feature embeddings: width stays `emb_dim`.
- `concat` stacks them side by side (`n_features * emb_dim`), then projects to `fusion_dim`.

In [8]:
from next_sequence import FusionStrategy

for strategy in FusionStrategy:
    m = NextSequenceModel(NextSequenceConfig(features=features, max_len=16, emb_dim=32,
                                             attn_heads=2, fusion_strategy=strategy))
    print(f"{strategy.value:7s} {sum(p.numel() for p in m.parameters()):>8,} parameters | fusion: {m.backbone.fusion}")

sum       18,155 parameters | fusion: SumFusion(
  (proj): Identity()
)
concat    22,283 parameters | fusion: ConcatFusion(
  (proj): Linear(in_features=128, out_features=32, bias=True)
)
